# Chapter 3. Advanced Vector Retrieval Strategies

The basic implementations of text embeddings and vector similarity search can produce insufficient retrieval accuracy and recall.

The embeddings generated from a user's query might not always align closely with those of documents containing the crucial information needed due to differences in terminology or context.

One strategy to improve the retrieval accuracy and recall is to *rewrite the query used to find relevant documents*.

The query-rewriting approach aims to bridge the gap between the user’s query and the information-rich documents by reformulating the query in a way that better aligns with the language and context of the target documents.

Another way to improve retrieval accuracy is by changing the document embedding strategy. Instead of embedding the exact text we plan to retrieve, we can embed content that better represents the document's meaning, such as more contextually relevant sections, synthetic questions, or paraphrased versions of the text to better capture key ideas and themes, resulting in more accurate and relevant retrieval outcomes.

There are two main strategies for improving document embedding:
- *Hypothetical question–embedding strategy*: With the hypothetical question–embedding strategy, we must determine the questions the information in the document can answer. We can use an LLM to generate hypothetical questions, or we can use the conversation history of our chatbot to come up with the questions a document can answer. When a user poses a question, the system computes the query’s embedding and searches for the nearest neighbors among the precomputed question embeddings.  The goal is to *locate questions that closely match and are semantically similar to the user question*. The system then retrieves the documents that contain the information that can answer these similar questions.
- *Parent document-embedding strategy*: The original document - referred to as the parent - is split into smaller units called *child chunks*, based on a fixed token count. Instead of embedding the entire parent document as a single unit, we compute a separate embedding for each child chunk. When a user submits a query, the system compares it against these child embeddings to find the most relevant matches. However, rather than returning only the matched chunk, the system retrieves the entire original parent document associated with it.


Other strategies to improve retrieval accuracy include:
- *Finetuning the embedding model*
- *Reranking strategies*
- *Metadata-based contextual filtering*
- *Hybrid retrieval (keyword + dense vector search)*

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

## Step-back Prompting

In [2]:
from dotenv import load_dotenv
import re
from typing import List
import pdfplumber
import requests
load_dotenv()

True

In [3]:
from utils.utils import chat, chunk_text, embed, neo4j_driver, num_tokens_from_string

The *step-back prompting* is a query-rewriting technique that aims to improve the accuracy of vector retrieval.

LLMs are an excellent fit for query-rewriting tasks.

In [4]:
stepback_sys_msg = """
You are an expert at world knowledge. Your task is to step back
and paraphrase a question to a more generic step-back question, which
is easier to answer. Here are a few examples

"input": "Could the members of The Police perform lawful arrests?"
"output": "what can the members of The Police do?"

"input": "Jan Sindel's was born in what country?"
"output": "what is Jan Sindel's personal history?"
"""


In [5]:
def generate_stepback(question: str):
    user_msg = f"""{question}"""
    step_back_question = chat(
        messages=[
            {"role": "system", "content": stepback_sys_msg},
            {"role": "user", "content": user_msg},
        ]
    )
    return step_back_question

In [6]:
question = "Which team did Thierry Audel play for from 2007 to 2008?"
step_back_question = generate_stepback(question)
print(f"Stepback results: {step_back_question}")

Stepback results: What is Thierry Audel's career history?


The step-back prompting transforms a specific user query into a more general question.

## Parent Document Retriever

The parent document retriever strategy involves dividing a large document into smaller sections, calculating embeddings for each section rather than the whole document, and using these embeddings to match user queries more accurately, ultimately retrieving the entire document for context-rich responses.

We need to split the PDF into parent documents and further divide those into child documents.

![Parent document retriever strategy](./imgs/parent-doc-graph.png)

In [7]:
remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch03-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)

In [8]:
text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [9]:
def split_text_by_titles(text):
    # A regular expression pattern for titles that 
    # matches lines starting with one or more digits, an optional uppercase letter,
    # followed by a dot, a space, and then up to 50 characters
    title_pattern = re.compile(r"(\n\d+[A-Z]?\. {1,3}.{0,60}\n)", re.DOTALL)
    titles = title_pattern.findall(text)

    # Split the text at these titles
    sections = re.split(title_pattern, text)
    sections_with_titles = []

    # Append the first section
    sections_with_titles.append(sections[0])
    # Iterate over the rest of sections
    for i in range(1, len(titles) + 1):
        section_text = sections[i * 2 - 1].strip() + "\n" + sections[i * 2].strip()
        sections_with_titles.append(section_text)

    return sections_with_titles


sections = split_text_by_titles(text)
print(f"Number of sections: {len(sections)}")

Number of sections: 9


The regular expression is based on the fact that sections in the text are organized as a numbered list, where each new section starts with a number and an optional character, followed by a dot and the section title.

We can count the number of tokens in each section to better understand their lengths.

In [10]:
for s in sections:
    print(num_tokens_from_string(s))

153
256
4156
556
2677
789
634
192
600


The 3rd section has over 4000 tokens, and we must split the sections into parent documents, where each document has at most 2000 tokens.

In [11]:
parent_chunks = []
for s in sections:
    parent_chunks.extend(chunk_text(s, chunk_size=2000, overlap=40))

In [12]:
for chunk in parent_chunks:
    print(num_tokens_from_string(chunk))

153
256
429
487
444
474
468
451
464
469
442
110
432
131
453
450
552
456
435
389
455
346
462
185
192
417
197


Instead of splitting the child chunks and importing them in a subsequent step, we will perform the splitting and the import in a single step.

In [15]:
# Cypher query to import the parent document strategy graph
cypher_import_query = """
MERGE (pdf:PDF {id: $pdf_id})

MERGE (p:Parent {id: $pdf_id + '-' + $id})
SET p.text = $parent

MERGE (pdf)-[:HAS_PARENT]->(p)
WITH p, $children AS children, $embeddings AS embeddings
UNWIND range(0, size(children) - 1) AS child_index
MERGE (c:Child {id: $pdf_id + '-' + $id + '-' + toString(child_index)})
SET c.text = children[child_index], c.embedding = embeddings[child_index]
MERGE (p)-[:HAS_CHILD]->(c);
"""

The above Cypher statement starts by merging a `PDF` node.

Next, it merges the `Parent` node using a unique ID. The `Parent` node is then linked to the `PDF` node through a `HAS_PARENT` relationship and has the `text` property set.

Lastly, it iterates over a list of child documents. It creates a `Child` node for each element in the list, sets the text and embedding properties, and links it to its `Parent` node with a `HAS_CHILD` relationship.

In [16]:
# Import the parent doc data into Neo4j
for i, chunk in enumerate(parent_chunks):
    child_chunks = chunk_text(chunk, chunk_size=500, overlap=20)
    embeddings = embed(child_chunks)

    # Add to neo4j
    neo4j_driver.execute_query(
        cypher_import_query,
        id=str(i),
        pdf_id="1709.00666",
        parent=chunk,
        children=child_chunks,
        embeddings=embeddings
    )

Each parent document chunk is divided into multiple child chunks using the `chunk_text` function.

The overall graph structure:
![Graph structure](./imgs/overall-graph-structure.png)

We can use the following Cypher statement to examine the generated graph structure:

```cypher
MATCH p=(pdf:PDF)-[:HAS_PARENT]->()-[:HAS_CHILD]->()
RETURN p LIMIT 50
```

which returns the following graph structure:
![Graph structure](./imgs/graph-snippet.png)

To ensure efficient comparison of document embeddings, we can add a vector index:

In [17]:
index_name = "parent"
neo4j_driver.execute_query("""
CREATE VECTOR INDEX parent IF NOT EXISTS
FOR (c:Child)
ON c.embedding
""")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000001FAE5936090>, keys=[])

### Retrieving Parent Document Strategy Data

To retrieve relevant documents from the graph, we must define the retrieval Cypher statement:

In [18]:
# Parent doc retrieval query
retrieval_query = """
CALL db.index.vector.queryNodes($index_name, $k * 4, $question_embedding)
YIELD node, score
MATCH (node)<-[:HAS_CHILD]-(parent)
WITH parent, max(score) AS score
RETURN parent.text AS text, score
ORDER BY score DESC
LIMIT toInteger($k)
"""

The above Cypher query starts by executing a vector-based search within a graph database to identify child nodes closely aligned with a specified question embedding.

We will retrieve `k * 4` documents in the initial vector search. This is because multiple similar child nodes from the vector search may belong to the same parent document. Therefore, it becomes crucial to deduplicate the parent documents.

**Without deduplication, the result set could include multiple entries for the same parent document, each corresponding to a different child node of that parent.**

To guarantee a final count of `k` unique parent documents, we start with a larger pool of `k * 4` child nodes, effectively creating a safety buffer. In the end of the query, we order the parent documents by their maximum similarity score and limit the results to `k` unique parent documents.

Next we can execute the retrieval query and examine the retrieved documents.

In [20]:
def parent_retrieval(question: str, k: int = 5) -> List[str]:
    question_embedding = embed([question])[0]

    similar_records, _, _ = neo4j_driver.execute_query(
        retrieval_query,
        question_embedding=question_embedding,
        k=k,
        index_name=index_name
    )

    return [record['text'] for record in similar_records]

In [21]:
documents = parent_retrieval(
    "Who was the Einsten's collaborator on sound reproduction system?"
)
for d in documents:
    print(d)
    print("=" * 20)

113B. Sound reproduction system with Rudolf Goldschmidt
Rudolf Goldschmidt was a German Engineer and inventor. He earned his engineering degree
in 1898 and PhD in 1906. He spent a decade working in England with major firms such as Crampton,
Arc works, Westinghouse etc. On returning back to Germany he joined Darmstadt T H University as a
professor. Goldschmidt was a prolific inventor. His first patent was for a bicycle gear while still an
engineering student. In 1908 he developed a rotating radio‐frequency machine, which was used as an
early radio transmitter. The transmitter was used in the first trans‐Atlantic radiotelegraphic link
between Germany and United States, opened on 19th June, 1914, with an exchange of telegrams
between Kaiser Wilhelm II and President Woodrow Wilson.
Figure 5: Einstein‐Goldschmidt design of a sound reproduction system.
In 1922 Goldschmidt approached Einstein for his expert opinion regarding one of his patents.
Thereafter, they kept in touch. Even after Einst

## Complete RAG Pipeline

The last piece is the answer-generating function.

In [22]:
answer_sys_msg = "You're an Einstein expert, but can only use the provided documents to respond to the questions."

def generate_answer(question: str, documents: List[str]):
    user_msg = f"""
    Use the following documents to answer the question that will follow:
    {documents}

    ---

    The question to answer using information only from the above documents: {question}
    """

    result = chat(
        messages=[
            {"role": "system", "content": answer_sys_msg},
            {"role": "user", "content": user_msg},
        ]
    )
    print("Response: ", result)

Of course we need to combine both the step-back prompting and the parent document retriever strategies to get the best results.

In [23]:
def rag_pipeline(question: str) -> str:
    stepback_prompt = generate_stepback(question)
    print(f"Stepback prompt: {stepback_prompt}")

    documents = parent_retrieval(stepback_prompt)

    answer = generate_answer(question, documents)
    return answer

In [24]:
# Test the complete RAG pipeline
rag_pipeline("Who was the Einsten's collaborator on sound reproduction system?")

Stepback prompt: Who worked with Einstein on scientific projects?
Response:  Einstein’s collaborator on the sound reproduction system was Rudolf Goldschmidt.
